In [ ]:
# SHAM_SECRETS_FALLBACK_V3 (2026-10-15)
# بعد ساعات من التدريب فشلت خلية الحفظ بـ NameError: '_sf_orig' / '_sf_creds'
# (الغلاف القديم V1 يعتمد على متغيرات عامة اختفت). هذه الخلية مستقلة تماماً:
# لا تعتمد على أي متغير عام، تضبط KAGGLE_USERNAME/KAGGLE_KEY في البيئة و
# ~/.kaggle/kaggle.json فوراً، وتلتقط أي خطأ من الأغلفة السابقة (حتى NameError)
# فترجع القيم من البيئة. آمنة إن أُعيد تشغيلها.
def _sham_secrets_fallback_v3():
    import glob, json, os

    def load_creds(_glob=glob, _json=json, _os=os):
        u, k = _os.environ.get("KAGGLE_USERNAME"), _os.environ.get("KAGGLE_KEY") or _os.environ.get("KAGGLE_API_TOKEN")
        if u and k:
            return {"username": u, "key": k}
        pats = ["/kaggle/input/sham-runtime-creds/**/kaggle.json", "/kaggle/input/**/kaggle.json",
                _os.path.expanduser("~/.kaggle/kaggle.json")]
        for pat in pats:
            for p in sorted(_glob.glob(pat, recursive=True)):
                try:
                    with open(p) as f:
                        d = _json.load(f)
                    if d.get("username") and d.get("key"):
                        return d
                except Exception:
                    pass
        return {}

    c = load_creds()
    if c:
        os.environ.setdefault("KAGGLE_USERNAME", c["username"])
        if not (os.environ.get("KAGGLE_KEY") or os.environ.get("KAGGLE_API_TOKEN")):
            os.environ["KAGGLE_KEY"] = c["key"]
            if str(c["key"]).startswith("KGAT_"):
                os.environ["KAGGLE_API_TOKEN"] = c["key"]
        try:
            kp = os.path.expanduser("~/.kaggle/kaggle.json")
            if not os.path.exists(kp):
                os.makedirs(os.path.dirname(kp), exist_ok=True)
                with open(kp, "w") as f:
                    json.dump({"username": c["username"], "key": c["key"]}, f)
                os.chmod(kp, 0o600)
        except Exception:
            pass
    print("SHAM_SECRETS_FALLBACK_V3: بيانات كاكل", "جاهزة في البيئة" if c else "غير موجودة")
    try:
        from kaggle_secrets import UserSecretsClient
    except Exception:
        return
    if getattr(UserSecretsClient.get_secret, "_sham_safe3", False):
        return

    def get_secret(self, label, *a, _prev=UserSecretsClient.get_secret, _load=load_creds, **k):
        try:
            return _prev(self, label, *a, **k)
        except BaseException as e:
            if isinstance(e, KeyboardInterrupt):
                raise
            if label == "GITHUB_TOKEN":
                return ""
            if label in ("KAGGLE_USERNAME", "KAGGLE_KEY"):
                d = _load()
                if d:
                    return d["username"] if label == "KAGGLE_USERNAME" else d["key"]
            raise

    get_secret._sham_safe3 = True
    get_secret._sham_safe2 = True  # يمنع V2 من إعادة التغليف لو أعيد تشغيلها
    get_secret._sham_safe = True   # ويمنع V1 كذلك
    UserSecretsClient.get_secret = get_secret

try:
    _sham_secrets_fallback_v3()
except Exception as _sf3_exc:
    print("SHAM_SECRETS_FALLBACK_V3 غير مفعّل:", type(_sf3_exc).__name__)


In [ ]:
# SHAM_QUIET_START_V4 (2026-10-15)
# المالك لاحظ أن اللون الأصفر/البرتقالي للدفتر يظهر حين يكتب الدفتر أخطاء/تحذيرات
# (stderr) ثم يفشل لاحقاً. هذه الخلية تمنع الأخطاء المتوقعة من مصدرها، ولا تغيّر
# أي إعداد للنموذج: (1) تتذكّر السر الذي رفضه كاكل (HTTP 400) فلا تعيد طلبه وتطبع
# الخطأ كل مرة، وتعطي KAGGLE_* من البيئة مباشرة دون طلب الخادم، (2) تضبط
# __main__.__spec__ = None حتى يعمل multiprocessing(spawn) بلا AttributeError،
# (3) تخفّض تحذيرات HuggingFace المتوقعة (طلبات بلا توكن، أشرطة التقدم).
def _sham_quiet_start_v4():
    import os, sys, logging, warnings
    m = sys.modules.get("__main__")
    if m is not None and getattr(m, "__spec__", "missing") == "missing":
        try:
            m.__spec__ = None
        except Exception:
            pass
    os.environ.setdefault("HF_HUB_VERBOSITY", "error")
    os.environ.setdefault("HF_HUB_DISABLE_PROGRESS_BARS", "1")
    os.environ.setdefault("TRANSFORMERS_VERBOSITY", "error")
    os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")
    warnings.filterwarnings("ignore", message=".*unauthenticated requests to the HF Hub.*")
    for n in ("huggingface_hub", "datasets", "fsspec", "urllib3"):
        logging.getLogger(n).setLevel(logging.ERROR)
    try:
        from kaggle_secrets import UserSecretsClient
    except Exception:
        return
    if getattr(UserSecretsClient.get_secret, "_sham_safe4", False):
        return
    failed = set()

    def get_secret(self, label, *a, _prev=UserSecretsClient.get_secret, _failed=failed, **k):
        env = {"KAGGLE_USERNAME": os.environ.get("KAGGLE_USERNAME"),
               "KAGGLE_KEY": os.environ.get("KAGGLE_KEY") or os.environ.get("KAGGLE_API_TOKEN")}
        if label in env and env[label]:
            return env[label]            # جاهزة من V3 / sham-runtime-creds: لا طلب ولا خطأ 400
        if label in _failed and label == "GITHUB_TOKEN":
            return ""
        try:
            return _prev(self, label, *a, **k)
        except KeyboardInterrupt:
            raise
        except BaseException:
            _failed.add(label)
            if label == "GITHUB_TOKEN":
                return ""
            raise

    for attr in ("_sham_safe4", "_sham_safe3", "_sham_safe2", "_sham_safe"):
        setattr(get_secret, attr, True)
    UserSecretsClient.get_secret = get_secret

try:
    _sham_quiet_start_v4()
    print("SHAM_QUIET_START_V4: مفعّل")
except Exception as _q4_exc:
    print("SHAM_QUIET_START_V4 غير مفعّل:", type(_q4_exc).__name__)


# 🧩 شام — أداة الاسترجاع والإصلاح وإعادة الدمج (منفصلة)

دفتر **جديد ومنفصل** (2026-10-16). يشغّل `repair_remerge/sham_repair_remerge.py`:
1. يسترجع الأساس `final_merged.pt` وكل نقاط `sham*` (اكتشاف تلقائي بقاعدة المهندس نفسها).
2. يقيس كل نقطة بميزان المهندس (نص/محادثة/وسائط) ويتخطّى التالف.
3. تدريب إصلاح قصير منخفض المعدّل على خليط المهارات (يُستأنف عبر الجلسات).
4. يدمج بـ `guarded_merge` نفسه وينشر إلى `sham-repair-remerge-checkpoint` فقط.

لا يلمس ملفات المهندس ولا نقاطه ولا `final_merged.pt`. آمن لإعادة التشغيل في أي وقت.

### 1) سحب كود شام من GitHub

In [ ]:
import os
import sys
import subprocess
from kaggle_secrets import UserSecretsClient

GITHUB_TOKEN = UserSecretsClient().get_secret("GITHUB_TOKEN")
REPO_URL = f"https://{GITHUB_TOKEN}@github.com/jonsnow-org/Ttbik.git"
BRANCH = "sham-main"   # أداة الإصلاح موجودة على فرع شام
CLONE_DIR = "/kaggle/working/Ttbik"

if not os.path.exists(CLONE_DIR):
    subprocess.run(["git", "clone", "--depth", "1", "--branch", BRANCH, REPO_URL, CLONE_DIR], check=True)
    print("تم سحب الكود الحقيقي من مستودع GitHub بنجاح.")
else:
    subprocess.run(["git", "-C", CLONE_DIR, "pull"], check=True)
    print("الكود موجود بالفعل في هذه الجلسة — تم سحب أي تحديثات جديدة عليه.")

# أمان: git يحفظ رابط الاستنساخ (بما فيه GITHUB_TOKEN) حرفياً داخل
# .git/config -- وهذا المجلد يبقى ضمن نتاج (Output) هذه الجلسة، الذي قد
# يُستخدَم لاحقاً كمُدخَل (Notebook Output) لجلسة أخرى، أو يُشارَك بأي شكل.
# نزع التوكن من الرابط المحفوظ فور نجاح الاستنساخ يمنع تسربه عبر هذا
# المسار تماماً (ثغرة حقيقية اكتشفتها المالكة، 2026-09-21).
subprocess.run(["git", "-C", CLONE_DIR, "remote", "set-url", "origin", "https://github.com/jonsnow-org/Ttbik.git"], check=True)

CODE_DIR = os.path.join(CLONE_DIR, "ai-system", "colab", "sham_small")
assert os.path.exists(os.path.join(CODE_DIR, "model.py")), f"لم يتم العثور على model.py داخل {CODE_DIR}"
sys.path.insert(0, CODE_DIR)
print("كود ShamSmall الحقيقي جاهز في:", CODE_DIR)


In [ ]:
try:
    import tokenizers
    print(f"مكتبة tokenizers متوفرة مسبقاً (نسخة {tokenizers.__version__}).")
except ImportError:
    subprocess.run(["pip", "install", "-q", "tokenizers"], check=True)
    print("تم تثبيت مكتبة tokenizers.")

try:
    import soundfile
    print("مكتبة soundfile متوفرة مسبقاً.")
except ImportError:
    subprocess.run(["pip", "install", "-q", "soundfile"], check=True)
    print("تم تثبيت مكتبة soundfile.")


### 2) التشغيل

In [ ]:
import os, sys, torch
sys.path.insert(0, os.path.join(CODE_DIR, "repair_remerge"))
os.environ.setdefault("KAGGLE_USERNAME", "jonsnowjonsnow")
import sham_repair_remerge as rr
device = "cuda" if torch.cuda.is_available() else "cpu"
print("الجهاز:", device, "| خطوات الإصلاح لكل نقطة:", rr.REPAIR_STEPS, "| المعدّل:", rr.REPAIR_LR)
res = rr.run(device=device)
print("دُمج:", res["changed"], "| النشر:", res["published"])
